# Analise de Tendencias de Consumo e Comportamento do Cliente

Este notebook responde **15 perguntas analiticas** sobre dados de compras e comportamento de clientes (3.900 transacoes), utilizando duas abordagens:
- **Pandas** (Python)
- **PostgreSQL** (query equivalente)

## Configuracao e Carregamento dos Dados

### Opcao 1: Carregamento via API do Kaggle (sem download manual)

O bloco abaixo baixa o dataset diretamente pela **API do Kaggle** para um diretorio temporario, sem necessidade de download manual. Para utilizar, crie um arquivo  na raiz do projeto com sua chave de API:



> Se preferir usar o arquivo CSV local ja disponivel no repositorio, **pule este bloco** e execute diretamente o bloco da Opcao 2 abaixo.

In [ ]:
import os
import io
import glob
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi
from dotenv import load_dotenv

# Carrega variaveis do arquivo .env
load_dotenv()

# Autentica via .env
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")  # Substitua pelo nome da variavel correta do seu .env ou sua chave de API Kaggle diretamente

api = KaggleApi()
api.authenticate()

DATASET = "sahilislam007/shopping-trends-and-customer-behaviour-dataset"

# Baixa e descompacta para diretorio temporario
api.dataset_download_files(DATASET, path="/tmp/shopping_trends", unzip=True, quiet=False)

# Lista arquivos baixados
arquivos = glob.glob("/tmp/shopping_trends/*")
print(arquivos)

# Carrega o dataset
df = pd.read_csv("/tmp/shopping_trends/Shopping Trends And Customer Behaviour Dataset.csv", index_col=0)
print(f"Dataset carregado via API: {df.shape[0]} registros, {df.shape[1]} colunas")

### Opcao 2: Carregamento local (arquivo CSV no repositorio)

Se o arquivo CSV ja esta na pasta do projeto, execute este bloco para carregar os dados diretamente.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Shopping Trends And Customer Behaviour Dataset.csv', index_col=0)

print(f'Base de compras: {df.shape[0]} registros, {df.shape[1]} colunas')
print(f'Colunas: {list(df.columns)}')
print(f'\nTipos de dados:\n{df.dtypes}')
print(f'\nValores nulos: {df.isnull().sum().sum()}')

In [ ]:
df.head()

In [ ]:
df.describe(include="all").T

---
## Pergunta 1: Quais sao as categorias de produtos com maior receita total e qual a participacao percentual de cada uma?

Entender a distribuicao de receita entre as categorias e identificar quais segmentos impulsionam o faturamento.

### Resposta com Pandas

In [ ]:
receita_categoria = (
    df.groupby('Category')['Purchase Amount (USD)']
    .agg(Receita_Total='sum', Ticket_Medio='mean', Qtd_Compras='count')
    .sort_values('Receita_Total', ascending=False)
)
receita_categoria['Participacao_%'] = (
    receita_categoria['Receita_Total'] / receita_categoria['Receita_Total'].sum() * 100
).round(2)
receita_categoria

### Resposta com PostgreSQL

```sql
SELECT "Category",
       SUM("Purchase Amount (USD)") AS receita_total,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       COUNT(*) AS qtd_compras,
       ROUND(SUM("Purchase Amount (USD)") * 100.0 /
             (SELECT SUM("Purchase Amount (USD)") FROM shopping_trends), 2) AS participacao_pct
FROM shopping_trends
GROUP BY "Category"
ORDER BY receita_total DESC;
```

### Analise

**Clothing** domina com **44,73%** da receita total ($104.264), seguido por **Accessories** (31,83%), **Footwear** (15,49%) e **Outerwear** (7,95%). Apesar da diferenca de volume, o ticket medio e muito similar entre categorias (~$57-60), indicando que a lideranca de Clothing vem do **volume de vendas** (1.737 transacoes) e nao de precos mais altos.

---
## Pergunta 2: Como o comportamento de compra difere entre homens e mulheres?

Identificar diferencas de consumo entre generos para direcionar estrategias de marketing.

### Resposta com Pandas

In [ ]:
genero = (
    df.groupby('Gender').agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        avaliacao_media=('Review Rating', 'mean'),
        compras_anteriores=('Previous Purchases', 'mean')
    ).round(2)
)
print(genero)
print()

# Distribuicao por categoria e genero
cat_genero = df.groupby(['Gender', 'Category']).size().unstack(fill_value=0)
cat_genero

### Resposta com PostgreSQL

```sql
SELECT "Gender",
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media,
       ROUND(AVG("Previous Purchases")::numeric, 2) AS compras_anteriores
FROM shopping_trends
GROUP BY "Gender";

-- Distribuicao por categoria e genero
SELECT "Gender", "Category", COUNT(*) AS qtd
FROM shopping_trends
GROUP BY "Gender", "Category"
ORDER BY "Gender", qtd DESC;
```

### Analise

O publico masculino representa **68%** das compras (2.652), porem o ticket medio e praticamente identico entre generos ($59,54 para homens vs. $60,25 para mulheres). As avaliacoes e o historico de compras anteriores tambem sao muito semelhantes, sugerindo que **a diferenca esta no volume de clientes e nao no comportamento individual**. A proporcao entre categorias se mantem consistente em ambos os generos.

---
## Pergunta 3: Como o valor de compra varia entre diferentes faixas etarias?

Segmentar clientes por idade para identificar os grupos que mais consomem e direcionar acoes comerciais.

### Resposta com Pandas

In [ ]:
bins = [17, 25, 35, 45, 55, 70]
labels = ['18-25', '26-35', '36-45', '46-55', '56-70']
df['Faixa_Etaria'] = pd.cut(df['Age'], bins=bins, labels=labels)

faixa_etaria = (
    df.groupby('Faixa_Etaria', observed=False).agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        valor_total=('Purchase Amount (USD)', 'sum'),
        avaliacao_media=('Review Rating', 'mean')
    ).round(2)
)
faixa_etaria

### Resposta com PostgreSQL

```sql
SELECT CASE
         WHEN "Age" BETWEEN 18 AND 25 THEN '18-25'
         WHEN "Age" BETWEEN 26 AND 35 THEN '26-35'
         WHEN "Age" BETWEEN 36 AND 45 THEN '36-45'
         WHEN "Age" BETWEEN 46 AND 55 THEN '46-55'
         ELSE '56-70'
       END AS faixa_etaria,
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       SUM("Purchase Amount (USD)") AS valor_total,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media
FROM shopping_trends
GROUP BY faixa_etaria
ORDER BY faixa_etaria;
```

### Analise

A faixa **56-70 anos** lidera em volume com **1.105 compras** (28,3% do total) e receita total de $65.256, demonstrando que clientes mais velhos sao os mais ativos na plataforma. O ticket medio, porem, e notavelmente **uniforme** entre todas as faixas ($59-61), indicando que a idade influencia a **frequencia de compra**, mas nao o **valor gasto por transacao**.

---
## Pergunta 4: Quais sao os 10 itens mais vendidos e qual o ticket medio de cada um?

Identificar os produtos mais populares e entender a faixa de preco de cada um.

### Resposta com Pandas

In [ ]:
top_itens = (
    df.groupby('Item Purchased').agg(
        qtd_vendas=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        receita_total=('Purchase Amount (USD)', 'sum')
    )
    .sort_values('qtd_vendas', ascending=False)
    .head(10)
    .round(2)
)
top_itens.index.name = 'Item'
top_itens

### Resposta com PostgreSQL

```sql
SELECT "Item Purchased",
       COUNT(*) AS qtd_vendas,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       SUM("Purchase Amount (USD)") AS receita_total
FROM shopping_trends
GROUP BY "Item Purchased"
ORDER BY qtd_vendas DESC
LIMIT 10;
```

### Analise

**Blouse** e **Pants** lideram com **171 vendas** cada, seguidos por **Jewelry** (171). A distribuicao e extremamente equilibrada entre os 25 itens (variando de 140 a 171 vendas), sugerindo uma base de clientes com **demanda diversificada**. O ticket medio varia pouco ($56-62), com **Dress** apresentando o maior valor ($62,17) e **Jacket** o menor ($56,74).

---
## Pergunta 5: Como as vendas se distribuem entre as estacoes do ano? Existe sazonalidade nas categorias?

Detectar padroes sazonais que possam orientar planejamento de estoque e campanhas.

### Resposta com Pandas

In [ ]:
sazonalidade = (
    df.groupby('Season').agg(
        qtd_compras=('Customer ID', 'count'),
        receita_total=('Purchase Amount (USD)', 'sum'),
        ticket_medio=('Purchase Amount (USD)', 'mean')
    ).round(2)
)
print(sazonalidade)
print()

# Quantidade de vendas por categoria em cada estacao
cat_sazonal = (
    df.groupby(['Season', 'Category'])
    .size()
    .unstack(fill_value=0)
)
cat_sazonal

### Resposta com PostgreSQL

```sql
SELECT "Season",
       COUNT(*) AS qtd_compras,
       SUM("Purchase Amount (USD)") AS receita_total,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio
FROM shopping_trends
GROUP BY "Season"
ORDER BY qtd_compras DESC;

-- Categorias por estacao
SELECT "Season", "Category", COUNT(*) AS qtd_compras
FROM shopping_trends
GROUP BY "Season", "Category"
ORDER BY "Season", qtd_compras DESC;
```

### Analise

**Spring** lidera em volume (999 compras), mas **Fall** tem o maior ticket medio ($61,56) e a maior receita ($60.018). As categorias mantem proporcoes semelhantes em todas as estacoes, indicando que **nao ha sazonalidade marcante** por categoria. A distribuicao equilibrada sugere um consumo estavel ao longo do ano.

---
## Pergunta 6: Clientes com assinatura gastam mais e avaliam melhor os produtos?

Avaliar se o modelo de assinatura gera maior engajamento e receita.

### Resposta com Pandas

In [ ]:
assinatura = (
    df.groupby('Subscription Status').agg(
        qtd_clientes=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        avaliacao_media=('Review Rating', 'mean'),
        compras_anteriores=('Previous Purchases', 'mean')
    ).round(2)
)
assinatura

### Resposta com PostgreSQL

```sql
SELECT "Subscription Status",
       COUNT(*) AS qtd_clientes,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media,
       ROUND(AVG("Previous Purchases")::numeric, 2) AS compras_anteriores
FROM shopping_trends
GROUP BY "Subscription Status";
```

### Analise

Apenas **27%** dos clientes possuem assinatura (1.053 de 3.900). Surpreendentemente, **nao ha diferenca significativa** entre assinantes e nao-assinantes: ticket medio ($59,49 vs. $59,87), avaliacao (3,74 vs. 3,75) e compras anteriores (26,08 vs. 25,08) sao praticamente identicos. Isso sugere que o programa de assinatura **nao esta gerando valor diferenciado**, representando uma oportunidade de melhoria.

---
## Pergunta 7: Qual o impacto de descontos e codigos promocionais no valor de compra?

Medir a efetividade das estrategias de desconto e promocao.

### Resposta com Pandas

In [ ]:
desconto_promo = (
    df.groupby(['Discount Applied', 'Promo Code Used']).agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        avaliacao_media=('Review Rating', 'mean')
    ).round(2)
)
desconto_promo

### Resposta com PostgreSQL

```sql
SELECT "Discount Applied", "Promo Code Used",
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media
FROM shopping_trends
GROUP BY "Discount Applied", "Promo Code Used"
ORDER BY ticket_medio DESC;
```

### Analise

Desconto e codigo promocional **sempre ocorrem juntos** — todas as 1.677 compras com desconto tambem usaram promo code. O impacto no valor e minimo: compras sem desconto tem ticket medio de **$60,13** vs. **$59,28** com desconto, uma diferenca de apenas $0,85. A taxa de uso de promocoes e de **43%**, mostrando boa adesao mas impacto limitado no ticket.

---
## Pergunta 8: Quais metodos de pagamento sao mais utilizados e qual o ticket medio associado?

Mapear preferencias de pagamento para otimizar gateways e experiencia de checkout.

### Resposta com Pandas

In [ ]:
pagamento = (
    df.groupby('Payment Method').agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        receita_total=('Purchase Amount (USD)', 'sum')
    )
    .sort_values('qtd_compras', ascending=False)
    .round(2)
)
pagamento['Participacao_%'] = (
    pagamento['qtd_compras'] / pagamento['qtd_compras'].sum() * 100
).round(2)
pagamento

### Resposta com PostgreSQL

```sql
SELECT "Payment Method",
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       SUM("Purchase Amount (USD)") AS receita_total,
       ROUND(COUNT(*) * 100.0 /
             (SELECT COUNT(*) FROM shopping_trends), 2) AS participacao_pct
FROM shopping_trends
GROUP BY "Payment Method"
ORDER BY qtd_compras DESC;
```

### Analise

Os metodos de pagamento sao distribuidos de forma **extremamente equilibrada**, variando de 15,69% (Bank Transfer) a 17,36% (PayPal). **Debit Card** tem o maior ticket medio ($60,92), enquanto **Venmo** tem o menor ($58,95). A diversificacao indica que os clientes nao tem forte preferencia, reforcando a importancia de **manter todos os gateways ativos**.

---
## Pergunta 9: Existe correlacao entre a avaliacao do produto e o valor gasto na compra?

Investigar se clientes que pagam mais avaliam melhor os produtos.

### Resposta com Pandas

In [ ]:
correlacao = df['Review Rating'].corr(df['Purchase Amount (USD)'])
print(f'Correlacao de Pearson: {correlacao:.4f}')
print()

# Media de gasto por faixa de avaliacao
df['Faixa_Avaliacao'] = pd.cut(
    df['Review Rating'],
    bins=[2.4, 3.0, 3.5, 4.0, 4.5, 5.01],
    labels=['2.5-3.0', '3.1-3.5', '3.6-4.0', '4.1-4.5', '4.6-5.0']
)
avaliacao_valor = (
    df.groupby('Faixa_Avaliacao', observed=False).agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean')
    ).round(2)
)
avaliacao_valor

### Resposta com PostgreSQL

```sql
-- Correlacao de Pearson
SELECT ROUND(CORR("Review Rating", "Purchase Amount (USD)")::numeric, 4) AS correlacao
FROM shopping_trends;

-- Media por faixa de avaliacao
SELECT CASE
         WHEN "Review Rating" <= 3.0 THEN '2.5-3.0'
         WHEN "Review Rating" <= 3.5 THEN '3.1-3.5'
         WHEN "Review Rating" <= 4.0 THEN '3.6-4.0'
         WHEN "Review Rating" <= 4.5 THEN '4.1-4.5'
         ELSE '4.6-5.0'
       END AS faixa_avaliacao,
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio
FROM shopping_trends
GROUP BY faixa_avaliacao
ORDER BY faixa_avaliacao;
```

### Analise

A correlacao de Pearson e **0,0308** — praticamente **nula**. O ticket medio por faixa de avaliacao varia apenas de $58,94 (notas 2.5-3.0) a $61,00 (notas 4.6-5.0), uma diferenca de apenas $2. Isso demonstra que o valor pago **nao influencia a satisfacao do cliente**, e que a percepcao de qualidade e independente do preco.

---
## Pergunta 10: Quais sao os 10 estados com maior receita e qual o ticket medio por estado?

Identificar os mercados geograficos mais relevantes para o negocio.

### Resposta com Pandas

In [ ]:
top_estados = (
    df.groupby('Location').agg(
        qtd_compras=('Customer ID', 'count'),
        receita_total=('Purchase Amount (USD)', 'sum'),
        ticket_medio=('Purchase Amount (USD)', 'mean')
    )
    .sort_values('receita_total', ascending=False)
    .head(10)
    .round(2)
)
top_estados

### Resposta com PostgreSQL

```sql
SELECT "Location",
       COUNT(*) AS qtd_compras,
       SUM("Purchase Amount (USD)") AS receita_total,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio
FROM shopping_trends
GROUP BY "Location"
ORDER BY receita_total DESC
LIMIT 10;
```

### Analise

**Montana** lidera com **96 compras** e $5.784 em receita, seguido por **Illinois** ($5.617) e **California** ($5.605). Destaque para **West Virginia** que, apesar de ter menos compras (81), possui o maior ticket medio entre o top 10 ($63,88). A receita e distribuida entre todos os 50 estados de forma relativamente uniforme.

---
## Pergunta 11: Clientes com maior frequencia de compras gastam mais por transacao?

Entender se compradores mais frequentes sao tambem os que mais gastam por compra.

### Resposta com Pandas

In [ ]:
ordem = ['Weekly', 'Bi-Weekly', 'Fortnightly', 'Monthly',
         'Quarterly', 'Every 3 Months', 'Annually']

frequencia = (
    df.groupby('Frequency of Purchases').agg(
        qtd_clientes=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        avaliacao_media=('Review Rating', 'mean'),
        compras_anteriores=('Previous Purchases', 'mean')
    ).round(2)
    .reindex(ordem)
)
frequencia

### Resposta com PostgreSQL

```sql
SELECT "Frequency of Purchases",
       COUNT(*) AS qtd_clientes,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media,
       ROUND(AVG("Previous Purchases")::numeric, 2) AS compras_anteriores
FROM shopping_trends
GROUP BY "Frequency of Purchases"
ORDER BY CASE "Frequency of Purchases"
    WHEN 'Weekly' THEN 1
    WHEN 'Bi-Weekly' THEN 2
    WHEN 'Fortnightly' THEN 3
    WHEN 'Monthly' THEN 4
    WHEN 'Quarterly' THEN 5
    WHEN 'Every 3 Months' THEN 6
    WHEN 'Annually' THEN 7
END;
```

### Analise

**Nao ha relacao clara** entre frequencia de compras e valor gasto. O ticket medio varia apenas entre $58,97 (Weekly) e $60,69 (Bi-Weekly). Nota-se que **Bi-Weekly** e **Fortnightly** representam a mesma frequencia (quinzenal), assim como **Quarterly** e **Every 3 Months** (trimestral) — uma inconsistencia nos dados que vale monitorar.

---
## Pergunta 12: Qual tipo de frete e mais escolhido e como se relaciona com o valor da compra?

Analisar preferencias de entrega e seu impacto na experiencia e no ticket medio.

### Resposta com Pandas

In [ ]:
frete = (
    df.groupby('Shipping Type').agg(
        qtd_compras=('Customer ID', 'count'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        avaliacao_media=('Review Rating', 'mean')
    )
    .sort_values('qtd_compras', ascending=False)
    .round(2)
)
frete['Participacao_%'] = (
    frete['qtd_compras'] / frete['qtd_compras'].sum() * 100
).round(2)
frete

### Resposta com PostgreSQL

```sql
SELECT "Shipping Type",
       COUNT(*) AS qtd_compras,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media,
       ROUND(COUNT(*) * 100.0 /
             (SELECT COUNT(*) FROM shopping_trends), 2) AS participacao_pct
FROM shopping_trends
GROUP BY "Shipping Type"
ORDER BY qtd_compras DESC;
```

### Analise

**Free Shipping** lidera com **17,31%** (675 compras), seguido por **Standard** (16,77%). O **2-Day Shipping** tem o maior ticket medio ($60,73), enquanto **Standard** tem o menor ($58,46). Curiosamente, **Standard** tem a melhor avaliacao media (3,82), sugerindo que expectativas alinhadas com a entrega contribuem mais para a satisfacao do que a velocidade.

---
## Pergunta 13: Quais sao as cores mais populares em cada estacao do ano?

Identificar tendencias de preferencia cromatica sazonal para orientar colecoes e estoque.

### Resposta com Pandas

In [ ]:
cores_estacao = (
    df.groupby(['Season', 'Color'])
    .size()
    .reset_index(name='Qtd')
    .sort_values(['Season', 'Qtd'], ascending=[True, False])
    .groupby('Season')
    .head(5)
    .reset_index(drop=True)
)
cores_estacao

### Resposta com PostgreSQL

```sql
WITH ranked AS (
    SELECT "Season", "Color", COUNT(*) AS qtd,
           ROW_NUMBER() OVER (
               PARTITION BY "Season" ORDER BY COUNT(*) DESC
           ) AS rank
    FROM shopping_trends
    GROUP BY "Season", "Color"
)
SELECT "Season", "Color", qtd
FROM ranked
WHERE rank <= 5
ORDER BY "Season", qtd DESC;
```

### Analise

Cada estacao possui preferencias cromaticas distintas: **Fall** prefere **Magenta** e **Yellow** (50 cada), **Spring** prefere **Olive** (52), **Summer** destaca **Silver** (59 vendas — o maior valor individual) e **Winter** favorece **Green** (50). A cor **Olive** aparece no top em mais de uma estacao, sendo uma escolha versatil.

---
## Pergunta 14: Clientes com mais compras anteriores sao mais propensos a ter assinatura?

Investigar se a fidelidade (compras acumuladas) se traduz em adesao ao programa de assinatura.

### Resposta com Pandas

In [ ]:
df['Faixa_Compras'] = pd.cut(
    df['Previous Purchases'],
    bins=[0, 10, 20, 30, 40, 50],
    labels=['1-10', '11-20', '21-30', '31-40', '41-50']
)

fidelidade = (
    df.groupby('Faixa_Compras', observed=False).agg(
        total_clientes=('Customer ID', 'count'),
        pct_assinantes=('Subscription Status',
                        lambda x: (x == 'Yes').mean() * 100),
        ticket_medio=('Purchase Amount (USD)', 'mean')
    ).round(2)
)
fidelidade

### Resposta com PostgreSQL

```sql
SELECT CASE
         WHEN "Previous Purchases" BETWEEN 1 AND 10 THEN '1-10'
         WHEN "Previous Purchases" BETWEEN 11 AND 20 THEN '11-20'
         WHEN "Previous Purchases" BETWEEN 21 AND 30 THEN '21-30'
         WHEN "Previous Purchases" BETWEEN 31 AND 40 THEN '31-40'
         ELSE '41-50'
       END AS faixa_compras,
       COUNT(*) AS total_clientes,
       ROUND(AVG(CASE WHEN "Subscription Status" = 'Yes'
                      THEN 1 ELSE 0 END) * 100, 2) AS pct_assinantes,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio
FROM shopping_trends
GROUP BY faixa_compras
ORDER BY faixa_compras;
```

### Analise

Existe uma **leve tendencia crescente**: clientes com 1-10 compras tem **23,47%** de taxa de assinatura, subindo para **29,46%** na faixa 31-40. Porem, o grupo 41-50 cai para 27,06%, quebrando o padrao. A correlacao e fraca, sugerindo que o historico de compras sozinho **nao e um forte preditor** de adesao a assinatura.

---
## Pergunta 15: Qual o perfil do cliente que mais gasta? (Segmentacao por quartis de gasto)

Criar segmentos de clientes por nivel de gasto e tracar o perfil de cada grupo, identificando oportunidades de upselling.

### Resposta com Pandas

In [ ]:
df['Segmento_Gasto'] = pd.qcut(
    df['Purchase Amount (USD)'],
    q=4,
    labels=['Economico', 'Moderado', 'Alto', 'Premium']
)

perfil = (
    df.groupby('Segmento_Gasto', observed=False).agg(
        faixa_valor=('Purchase Amount (USD)',
                     lambda x: f'${x.min():.0f} - ${x.max():.0f}'),
        ticket_medio=('Purchase Amount (USD)', 'mean'),
        idade_media=('Age', 'mean'),
        pct_masculino=('Gender',
                       lambda x: (x == 'Male').mean() * 100),
        pct_assinante=('Subscription Status',
                       lambda x: (x == 'Yes').mean() * 100),
        pct_desconto=('Discount Applied',
                      lambda x: (x == 'Yes').mean() * 100),
        avaliacao_media=('Review Rating', 'mean'),
        compras_anteriores=('Previous Purchases', 'mean'),
        categoria_top=('Category', lambda x: x.mode()[0])
    ).round(2)
)
perfil

### Resposta com PostgreSQL

```sql
WITH quartis AS (
    SELECT *,
           NTILE(4) OVER (ORDER BY "Purchase Amount (USD)") AS quartil
    FROM shopping_trends
)
SELECT CASE quartil
         WHEN 1 THEN 'Economico'
         WHEN 2 THEN 'Moderado'
         WHEN 3 THEN 'Alto'
         WHEN 4 THEN 'Premium'
       END AS segmento,
       MIN("Purchase Amount (USD)") || ' - ' ||
       MAX("Purchase Amount (USD)") AS faixa_valor,
       ROUND(AVG("Purchase Amount (USD)")::numeric, 2) AS ticket_medio,
       ROUND(AVG("Age")::numeric, 1) AS idade_media,
       ROUND(AVG(CASE WHEN "Gender" = 'Male'
                      THEN 1 ELSE 0 END) * 100, 2) AS pct_masculino,
       ROUND(AVG(CASE WHEN "Subscription Status" = 'Yes'
                      THEN 1 ELSE 0 END) * 100, 2) AS pct_assinante,
       ROUND(AVG(CASE WHEN "Discount Applied" = 'Yes'
                      THEN 1 ELSE 0 END) * 100, 2) AS pct_desconto,
       ROUND(AVG("Review Rating")::numeric, 2) AS avaliacao_media,
       ROUND(AVG("Previous Purchases")::numeric, 1) AS compras_anteriores
FROM quartis
GROUP BY quartil
ORDER BY quartil;
```

### Analise

A segmentacao revela que os quatro quartis tem perfis **notavelmente similares**: idade media (~44 anos), proporcao masculina (~67-70%), taxa de assinatura (~27%), e avaliacao (~3,7). A unica diferenca real e o valor gasto em si (de $29,52 no Economico a $91,09 no Premium). **Clothing** e a categoria favorita em todos os segmentos. Essa homogeneidade sugere que o valor de compra e influenciado mais pelo **produto escolhido** do que pelo perfil demografico do cliente.

---
## Conclusao

Esta analise explorou 15 dimensoes dos dados de comportamento de compra de clientes:

| Tema | Perguntas | Principal Descoberta |
|------|-----------|---------------------|
| Receita e Produtos | 1, 4 | Clothing domina em volume; itens muito equilibrados |
| Demografia | 2, 3 | Genero e idade nao impactam valor de compra |
| Sazonalidade | 5, 13 | Distribuicao uniforme; preferencias de cor variam |
| Fidelidade e Assinatura | 6, 14 | Assinatura nao gera diferencial mensuravel |
| Promocoes e Descontos | 7 | Impacto minimo no valor de compra |
| Pagamento e Frete | 8, 12 | Distribuicao equilibrada entre opcoes |
| Correlacoes | 9, 11 | Avaliacao e frequencia independentes do valor |
| Segmentacao | 15 | Perfis muito homogeneos entre segmentos de gasto |
| Geografia | 10 | Receita pulverizada entre estados |

> **Insight geral:** O dataset apresenta distribuicoes surpreendentemente uniformes em quase todas as dimensoes, sugerindo que as variaveis demograficas e comportamentais analisadas **nao sao fortes preditoras** do valor de compra. Isso aponta para a necessidade de coletar dados adicionais (historico de navegacao, tempo no site, campanhas vistas) para modelagens preditivas mais eficazes.